# House Price Prediction Project

In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:

df=pd.read_csv('houses_improved.csv')
df.head()

,Number_of_Rooms,Site_Area_sqm,Built_Area_sqm,Property_Years,Construction_Materials,Housing_Typology,Land_Value_Grading,Proximity_to_CBD_km,Proximity_to_Bus_Station_km,Type_of_Nearest_Road,Proximity_to_Schools_km,Price_ETB
0,3,402,275,18,Concrete,Semi-detached,Low,0.66,0.39,Gravel,1.51,2453165
1,6,111,72,5,Concrete,Semi-detached,High,0.12,1.38,Asphalt,2.04,2006785
2,4,417,306,2,Concrete,Condominium,Medium,5.00,2.21,Gravel,2.71,2164504
3,4,179,72,4,Mud&Wood,Condominium,Low,4.03,1.08,Asphalt,1.30,400087
4,3,315,147,12,Concrete,Semi-detached,Medium,3.95,1.26,Asphalt,2.89,1366641


In [3]:
#categorical_columns = df.select_dtypes(include=['object']).columns
categorical_cols = [
    "Construction_Materials",
    "Housing_Typology",
    "Land_Value_Grading",
    "Type_of_Nearest_Road",
]

# Dropdown choices for the UIs, taken straight from the data
#category_choices = {col: sorted(df[col].unique().tolist()) for col in categorical_cols}

In [4]:
# One-Hot Encoding (no drop_first, matches the notebook)
df_ohe = pd.get_dummies(df, columns=categorical_cols, dtype=int)
X_ohe = df_ohe.drop(columns=["Price_ETB"])
y = df_ohe["Price_ETB"]

# Label Encoding (FIX: capture each encoder into label_encoders)

df_le = df.copy()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col])
    label_encoders[col] = le
X_le = df_le.drop(columns=["Price_ETB"])


In [5]:

# Train/test splits
# ---------------------------------------------------------------------------
X_tr_ohe, X_te_ohe, y_train, y_test = train_test_split(X_ohe, y, test_size=0.2, random_state=42)
X_tr_le, X_te_le, _, _ = train_test_split(X_le, y, test_size=0.2, random_state=42)

encodings = {
    "One-Hot Encoding": (X_tr_ohe, X_te_ohe, X_ohe.columns),
    "Label Encoding": (X_tr_le, X_te_le, X_le.columns),
}

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "None (Unscaled)": None,
}


def get_models():
    return {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0),
        "Lasso Regression": Lasso(alpha=100.0),
        "Support Vector Regression (SVR)": SVR(kernel="rbf", C=1000),
        "Random Forest Regressor (RFR)": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting (GBM)": GradientBoostingRegressor(n_estimators=100, random_state=42),
        "Perceptron Regressor (MLP)": MLPRegressor(hidden_layer_sizes=(100,), max_iter=3000, random_state=42),
    }



In [6]:
# Train every combination
# ---------------------------------------------------------------------------
all_results = []
trained_pipelines = {}

for enc_name, (X_tr, X_te, feature_cols) in encodings.items():
    for scale_name, scaler_obj in scalers.items():
        if scaler_obj is not None:
            scaler = scaler_obj.__class__()
            X_tr_proc = scaler.fit_transform(X_tr)
            X_te_proc = scaler.transform(X_te)
        else:
            scaler = None
            X_tr_proc = X_tr.values
            X_te_proc = X_te.values

        for model_name, model in get_models().items():
            try:
                model.fit(X_tr_proc, y_train)
                predictions = model.predict(X_te_proc)

                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                r2 = r2_score(y_test, predictions)

                key = f"{model_name} | {enc_name} | {scale_name}"

                trained_pipelines[key] = {
                    "model": model,
                    "scaler": scaler,
                    "encoding": enc_name,
                    "columns": feature_cols,
                    "algorithm": model_name,
                    "scaler_name": scale_name,
                }

                all_results.append({
                    "Pipeline Key": key,
                    "Algorithm": model_name,
                    "Encoding": enc_name,
                    "Scaler": scale_name,
                    "R² Score": round(r2, 4),
                    "RMSE (ETB)": round(rmse, 2),
                    "MAE (ETB)": round(mae, 2),
                })

                print(f"OK  {key}  (R²={r2:.4f})")

            except Exception as e:
                print(f"FAIL  {model_name} | {enc_name} | {scale_name}  -> {e}")

results_df = (
    pd.DataFrame(all_results)
    .sort_values(by="R² Score", ascending=False)
    .reset_index(drop=True)
)

best_pipeline_key = results_df.iloc[0]["Pipeline Key"]

best_pipeline = trained_pipelines[best_pipeline_key]

print("\n")
print("=" * 60)
print(" BEST PIPELINE")
print("=" * 60)
print(best_pipeline_key)
print(f"R² Score : {results_df.iloc[0]['R² Score']}")



OK  Linear Regression | One-Hot Encoding | StandardScaler  (R²=0.8086)
OK  Ridge Regression | One-Hot Encoding | StandardScaler  (R²=0.8089)
OK  Lasso Regression | One-Hot Encoding | StandardScaler  (R²=0.8087)
OK  Support Vector Regression (SVR) | One-Hot Encoding | StandardScaler  (R²=-0.0723)
OK  Random Forest Regressor (RFR) | One-Hot Encoding | StandardScaler  (R²=0.8546)
OK  Gradient Boosting (GBM) | One-Hot Encoding | StandardScaler  (R²=0.9036)


C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(


OK  Perceptron Regressor (MLP) | One-Hot Encoding | StandardScaler  (R²=-2.6248)
OK  Linear Regression | One-Hot Encoding | MinMaxScaler  (R²=0.8086)
OK  Ridge Regression | One-Hot Encoding | MinMaxScaler  (R²=0.8104)
OK  Lasso Regression | One-Hot Encoding | MinMaxScaler  (R²=0.8088)
OK  Support Vector Regression (SVR) | One-Hot Encoding | MinMaxScaler  (R²=-0.0769)
OK  Random Forest Regressor (RFR) | One-Hot Encoding | MinMaxScaler  (R²=0.8547)
OK  Gradient Boosting (GBM) | One-Hot Encoding | MinMaxScaler  (R²=0.9036)


C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(


OK  Perceptron Regressor (MLP) | One-Hot Encoding | MinMaxScaler  (R²=-2.4951)
OK  Linear Regression | One-Hot Encoding | None (Unscaled)  (R²=0.8086)
OK  Ridge Regression | One-Hot Encoding | None (Unscaled)  (R²=0.8089)
OK  Lasso Regression | One-Hot Encoding | None (Unscaled)  (R²=0.8087)
OK  Support Vector Regression (SVR) | One-Hot Encoding | None (Unscaled)  (R²=-0.0312)
OK  Random Forest Regressor (RFR) | One-Hot Encoding | None (Unscaled)  (R²=0.8548)
OK  Gradient Boosting (GBM) | One-Hot Encoding | None (Unscaled)  (R²=0.9040)


...

C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(


OK  Perceptron Regressor (MLP) | Label Encoding | StandardScaler  (R²=-2.6648)
OK  Linear Regression | Label Encoding | MinMaxScaler  (R²=0.6835)
OK  Ridge Regression | Label Encoding | MinMaxScaler  (R²=0.6856)
OK  Lasso Regression | Label Encoding | MinMaxScaler  (R²=0.6837)
OK  Support Vector Regression (SVR) | Label Encoding | MinMaxScaler  (R²=-0.0709)
OK  Random Forest Regressor (RFR) | Label Encoding | MinMaxScaler  (R²=0.8361)
OK  Gradient Boosting (GBM) | Label Encoding | MinMaxScaler  (R²=0.8967)


C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(


OK  Perceptron Regressor (MLP) | Label Encoding | MinMaxScaler  (R²=-2.5977)
OK  Linear Regression | Label Encoding | None (Unscaled)  (R²=0.6835)
OK  Ridge Regression | Label Encoding | None (Unscaled)  (R²=0.6836)
OK  Lasso Regression | Label Encoding | None (Unscaled)  (R²=0.6836)
OK  Support Vector Regression (SVR) | Label Encoding | None (Unscaled)  (R²=-0.0293)
OK  Random Forest Regressor (RFR) | Label Encoding | None (Unscaled)  (R²=0.8360)
OK  Gradient Boosting (GBM) | Label Encoding | None (Unscaled)  (R²=0.8961)
OK  Perceptron Regressor (MLP) | Label Encoding | None (Unscaled)  (R²=0.3429)


 BEST PIPELINE
Gradient Boosting (GBM) | One-Hot Encoding | None (Unscaled)
R² Score : 0.904


C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (3000) reached and the optimization hasn't converged yet.
  warnings.warn(


Continue by inserting the preprocessing, model training, evaluation, and Gradio UI sections discussed.

In [7]:
# ---------------------------------------------------------------------------
# Save everything the apps need
# ---------------------------------------------------------------------------
bundle = {
    "trained_pipelines": trained_pipelines,
    "results_df": results_df,
    "categorical_cols": categorical_cols,
    "label_encoders": label_encoders,
    "best_pipeline_key": best_pipeline_key,
    "category_choices": category_choices,
}

joblib.dump(bundle, "pipeline_bundle.pkl")
print("\nSaved pipeline_bundle.pkl")
print(f"Best pipeline: {best_pipeline_key}")
print(f"Best R²: {results_df.iloc[0]['R² Score']}")


Saved pipeline_bundle.pkl
Best pipeline: Gradient Boosting (GBM) | One-Hot Encoding | None (Unscaled)
Best R²: 0.904
